# AOU-0.5 — Tier 0 MECHANISM PROBE. Phase M3 / Wave 1 validation.

**Purpose.** Isolate the **2048-partition executor write/finalize path** under `spark.executor.cores=1` / `executor.memory=5g` with **ZERO source read** and **ZERO QC**. A synthetic `range_matrix_table(50_000, 2_000).repartition(2048)` MT is written via `mt.checkpoint(...)` — the exact Hail write-finalization step that produced the m3-W1 empty-MT catastrophe (0×0 schema-only MTs, `_SUCCESS` written on driver-side task accounting without validating executor row-group payloads).

Partition count is **FIXED** at 2048 in the production path (`naive_coalesce(2048)` + `repartition(2048)` in `aou_ld_panel.py`) regardless of any interval filter, so this synthetic 2048-partition write is a FAITHFUL reproduction of the catastrophe-triggering partition profile while costing almost nothing.

**HONEST CAVEAT (load-bearing).** A Tier-0 PASS rules out **only** the pure-write-path failure mode — it does NOT rule out memory-pressure truncation during the real QC pipeline (sample_qc / variant_qc / split_multi over genome-scale data). **Never label a Tier-0 pass "validated"; label it "no cheap failure mode reproduced, escalating to the real test."** Only the chr22 (Tier 2) / full-genome fire reaches genome-scale memory pressure.

**Cluster (2026-06-02, web-verified — see runbook §0).** Create a **"Hail Genomics Analysis" Dataproc cluster** (Verily `software-framework=HAIL`) — Hail is **pre-installed + YARN-wired**, so NO pip. Do **NOT** use the generic **"JupyterLab Spark cluster"** (that is `software-framework=NONE` — no Hail). Workers: **4× n2-standard-16 = 64 vCPU** (Verily offers only `n2-standard`, no `n1-highmem`; the 5 GB/executor profile is preserved by the Cell 1a `PYSPARK_SUBMIT_ARGS` lever regardless of node family), **NON-preemptible** (spot would muddy the kill-interrupted-write hypothesis), SHARED with the Tier 1 nano fire. Image **Dataproc 2.2 (Spark 3.5)** if a selector appears. **After Cell 1b, run the YARN-verify check** (`sc.master` must start with `yarn`, not `local`; executors > 1; exec mem `5g`) before trusting the probe.

**Cost.** ~$1-3. **NOT part of the compute-free AOU-0 precheck** (this is a compute fire). Carter holds the trigger.

**Gate A decision rule.**
- **PASS** (synthetic MT writes with `count_rows>0`/`count_cols>0` and few-MB entries) → proceed to **Tier 1 (nano) on the SAME cluster** (AOU-1 with `INTERVAL = "chr22:16000000-18000000"`). Remember: PASS = "no cheap failure mode reproduced".
- **FAIL** (empty MT / `_assert_checkpoint_nonempty` raises / sub-floor entries) → the catastrophe is **ruled IN at the pure-write path** → pivot Wave 2 to **1000G AFR** (only ~$1-3 spent). The `_forensics/probe_capture.json` records the `_SUCCESS`-mtime-vs-part-mtimes hypothesis flag (`[[feedback_w1_catastrophe_hypothesis_distinguisher]]`).

**Cross-references:**
- `TIERED-VALIDATION-RUNBOOK.md` — §0 cluster provisioning + Gate A/B/C decision tree
- `.planning/notebooks/AOU-1-chr22-smoke_template.ipynb` — Tier 1 + Tier 2
- `.planning/debug/m3-W1-empty-mt-catastrophe.md` — catastrophe forensics
- `[[feedback_aou_dataproc_pyspark_submit_args]]` — the Cell 1a lever
- `[[feedback_aou_success_marker_not_evidence_of_data]]`
- `[[feedback_hail_checkpoint_contract_violation]]`

In [ ]:
# Cell 1a — Force Spark executor resources + requester-pays GCS billing at the spark-submit boundary.
# CANONICAL PATTERN (feedback_aou_dataproc_pyspark_submit_args, baked 2026-05-12): on AoU's
# Dataproc + YARN cluster, hl.init(spark_conf=dict) is silently overridden; PYSPARK_SUBMIT_ARGS
# injected BEFORE any pyspark/hail import IS honored (spark-submit boundary = highest precedence).
# This cell MUST run before any other pyspark/hail import. Pairs with naive_coalesce(2048)+
# repartition(2048) in aou_ld_panel.py (DEC-2026-05-04-01 v8 partition-explosion OOM remediation).
#
# REQUESTER-PAYS (added 2026-06-02): the controlled WGS bucket vwb-aou-datasets-controlled is
# requester-pays, and the migrated Verily Hail cluster does NOT pre-set the GCS-connector billing
# project (classic AoU did, in spark-defaults). CUSTOM mode scopes the userProject billing header
# to ONLY that bucket (the non-RP output bucket is untouched). GOTCHAS: bucket NAME only, NO gs://
# prefix (prefix => silent match failure); project = $GOOGLE_PROJECT; must be set pre-SparkContext.
# If reads still 400 "requester pays ... no user project", the env forced the newer STORAGE_CLIENT
# => add: --conf spark.hadoop.fs.gs.client.type=HTTP_API_CLIENT (RP honored only on HTTP_API_CLIENT).
import os
_proj = os.environ.get("GOOGLE_PROJECT", "wb-perky-corn-6639")
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--conf spark.executor.cores=1 "
    "--conf spark.executor.memory=5g "
    "--conf spark.driver.cores=1 "
    "--conf spark.hadoop.fs.gs.requester.pays.mode=CUSTOM "
    "--conf spark.hadoop.fs.gs.requester.pays.buckets=vwb-aou-datasets-controlled "
    f"--conf spark.hadoop.fs.gs.requester.pays.project.id={_proj} "
    "pyspark-shell"
)
print("PYSPARK_SUBMIT_ARGS set:", os.environ["PYSPARK_SUBMIT_ARGS"])

In [ ]:
# Cell 1a' — Portable sys.path (HOME-relative; works on any cluster home dir).
# The migrated Verily Hail Dataproc cluster runs as user 'dataproc' (HOME=/home/dataproc),
# NOT 'jupyter'. Cell 1b's hard-coded /home/jupyter/... insert is a harmless no-op there;
# this HOME-relative insert makes `from aou_ld_panel import ...` resolve regardless of home,
# given the repo was cloned to ~/coloc_analysis per the runbook.
import sys, os
sys.path.insert(0, os.path.expanduser("~/coloc_analysis/src/python"))
print("sys.path[0] =", sys.path[0])

In [ ]:
# Cell 1a'' — env-var preflight (setdefault). The migrated Verily Hail Dataproc cluster does NOT
# auto-set WORKSPACE_BUCKET or the WGS input path (classic AoU did; only GOOGLE_PROJECT is auto-set).
# setdefault uses the platform value when present, else binds the verified migrated values.
# (WGS path verified readable 2026-06-02 via gsutil ls -u $GOOGLE_PROJECT; bucket is requester-pays
# — handled by Cell 1a's CUSTOM RP config.)
import os
os.environ.setdefault("WORKSPACE_BUCKET", "gs://rw-migration-aou-rw-476cdac2")
os.environ.setdefault("WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH",
                      "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt")
print("WORKSPACE_BUCKET =", os.environ["WORKSPACE_BUCKET"])
print("WGS path         =", os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"])
print("GOOGLE_PROJECT   =", os.environ.get("GOOGLE_PROJECT", "(unset!)"))

In [ ]:
# Cell 1b — Initialize Hail with spark_conf threading + verify executor.cores=1.
# Calls hl.init directly (NOT through init_hail wrapper) because the wrapper's
# spark_conf path is known broken on AoU YARN; the PYSPARK_SUBMIT_ARGS lever
# from Cell 1a is what actually binds the conf. The spark_conf dict here is
# belt-and-suspenders for portability — preserves the conf-by-dict path for
# environments where it works (local Spark, standalone clusters), while AoU
# binds via the env-var lever.
import sys
import os
# Portable sys.path (HOME-relative): migrated Verily Hail Dataproc runs as
# user 'dataproc' (HOME=/home/dataproc), not 'jupyter'. expanduser('~') resolves
# regardless of cluster home dir, given the repo was cloned to ~/coloc_analysis.
sys.path.insert(0, os.path.expanduser("~/coloc_analysis/src/python"))
import hail as hl
hl.init(
    default_reference="GRCh38",
    log="/tmp/hail.log",
    quiet=True,
    spark_conf={
        "spark.executor.cores": "1",
        "spark.executor.memory": "5g",
        "spark.driver.cores": "1",
    },
)

# Tier-0 needs only the two Task-1 helpers (no cohort load):
from aou_ld_panel import _assert_checkpoint_nonempty, _capture_catastrophe_forensics, _interval_scaled_du_floor

sc_conf = hl.spark_context().getConf()
cores = sc_conf.get('spark.executor.cores')
assert cores == '1', (
    f"PYSPARK_SUBMIT_ARGS lever did not bind — got cores={cores}, expected '1'. "
    f"DO NOT proceed — the v8 partition-explosion OOM config is NOT live. "
    f"Action: Kernel menu → Restart Kernel; then re-fire Cell 1a + Cell 1b."
)
print('=== HAIL INIT (Tier 0 probe) ===')
print(f'  Hail version          : {hl.__version__}')
print(f'  spark.executor.cores  : {cores}  OK')
print(f'  spark.executor.memory : {sc_conf.get("spark.executor.memory")}')
print(f'  spark.driver.cores    : {sc_conf.get("spark.driver.cores")}')
print(f'  WORKSPACE_BUCKET      : {os.environ["WORKSPACE_BUCKET"]}')


In [ ]:
# Probe cell — synthetic 2048-partition write under cores=1/5g.
# ZERO source read, ZERO QC: this exercises ONLY the Hail checkpoint
# write/finalize path that produced the m3-W1 empty-MT catastrophe.
import subprocess

# Synthetic MT: 50k rows x 2k cols, repartitioned to the FIXED 2048-partition
# profile the production path always uses (naive_coalesce(2048) + repartition(2048)).
mt = hl.utils.range_matrix_table(n_rows=50_000, n_cols=2_000).repartition(2048)
mt = mt.annotate_entries(x=hl.rand_norm())
_uri = f"{os.environ['WORKSPACE_BUCKET'].rstrip('/')}/ld/_probe_synthetic.mt"
mt = mt.checkpoint(_uri, overwrite=True)

try:
    # HARD GATE (UNCHANGED Track-4 guard): raises if the checkpoint read-back
    # is empty — the catastrophe signature.
    _assert_checkpoint_nonempty(mt, _uri, phase='probe')

    # Few-MB du SOFT-floor (NOT 50 MB) on entries/rows/parts/. For a synthetic
    # 50k x 2k MT the entries payload is a few MB; we scale the floor for a
    # nano-span so it does not false-positive. The count>0 gate above is the real test.
    _entries = _uri.rstrip('/') + '/entries/rows/parts/'
    _floor = _interval_scaled_du_floor('chr22:16000000-18000000', base_floor_bytes=50_000_000)
    _r = subprocess.run(['gsutil', 'du', '-s', _entries], capture_output=True, text=True)
    if _r.returncode == 0 and _r.stdout.split():
        _size = int(_r.stdout.split()[0])
        # SOFT: print, do not hard-fail below floor — the count>0 assertion is the gate.
        _flag = 'OK' if _size > _floor else 'SOFT-WARN (below few-MB floor — inspect)'
        print(f'{_flag}: {_entries} = {_size:,} bytes (soft-floor {_floor:,})')
    else:
        print(f'SOFT-WARN: could not du {_entries} (returncode={_r.returncode}); count>0 gate already passed.')
    print('GATE A PASS: synthetic 2048-partition write produced a populated MT.')
    print('RIGOR: this is \'no cheap failure mode reproduced\' — NOT \'validated\'. Escalate to Tier 1.')
except Exception:
    # Best-effort forensic capture (never raises) BEFORE re-raising. Records the
    # _SUCCESS-mtime-vs-part-mtimes distinguisher to _forensics/probe_capture.json.
    _capture_catastrophe_forensics(_uri, phase='probe')
    raise


## Tier 0 (Gate A) output: a synthetic probe MT at `gs://${WORKSPACE_BUCKET}/ld/_probe_synthetic.mt` + (on FAIL) a `gs://${WORKSPACE_BUCKET}/ld/_forensics/probe_capture.json`.

**Gate A decision:**
- **PASS** → the pure 2048-partition write/finalize path is healthy on this platform. Proceed to **Tier 1 (nano)** on the SAME 64-vCPU cluster: open `AOU-1-chr22-smoke_template.ipynb`, set `INTERVAL = "chr22:16000000-18000000"`, and fire. **PASS = "no cheap failure mode reproduced, escalating to the real test"** — NOT "validated".
- **FAIL** (the cell raised) → the catastrophe is **ruled IN at the pure-write path**. Do NOT spin the expensive Tier 2 / full-genome cluster. Pivot Wave 2 to the **1000G AFR** substrate (only ~$1-3 spent). Inspect `_forensics/probe_capture.json` for the `hypothesis_flag` (`hail_finalize_on_empty` vs `kill_interrupted_write`) per `[[feedback_w1_catastrophe_hypothesis_distinguisher]]`.

Delete the Dataproc cluster the MOMENT the tier finishes (orphan-kernel billing watchpoint, `[[feedback_aou_websocket_drop_zombie_pattern]]`). Carter holds the trigger for every launch.
